# Module 12 — Automatic Article Validation

**Second stage of Module 12.** Uses the same `CAMPAIGNS` + positional selection convention as the other DeepTuneIO modules. Article figures and article modes are resolved from the Golden Reference and mapped to Module 10 result access modes when terminology differs. For `DLIOv1/lustre_ost`, the campaign may contain several Lustre stripe counts and each published point is matched against its own `stripe_count`.


In [1]:
from pathlib import Path
import importlib.util, sys
MODULE_ROOT=(Path.cwd()/"Module").resolve()
if str(MODULE_ROOT) not in sys.path: sys.path.insert(0,str(MODULE_ROOT))
SCRIPT=MODULE_ROOT/"12_validate_campaign.py"
spec=importlib.util.spec_from_file_location("module12_auto",SCRIPT)
module12_auto=importlib.util.module_from_spec(spec); spec.loader.exec_module(module12_auto)
from Article_Validation.campaign import article_application_name, infer_filesystem, resolve_stripe_scope
print("Automatic validation version:",module12_auto.VERSION)


Automatic validation version: 1.5.2.1


In [23]:
# ============================================================
# Campaign selection — standard DeepTuneIO convention
# ============================================================
PROJECT_ROOT = Path(r"F:\NoteBooks\CORA_DR_UAB")
ARTICLE_DIR = "Article_01_Parallel_IO_Analysis"
DATA_ROOT = PROJECT_ROOT / ARTICLE_DIR / "Data"

CAMPAIGNS = {
    "DeepGalaxy": {
        "DG_bw512_f3c5x64": {
            "scenarios": ["lustre_1ost", "lustre_2ost", "lustre_4ost", "nfs_SnGPU_e1"],
            "file_format": ".hdf5",
        },
    },
    "DLIOv1": {
        "HDF5_64bs": {"scenarios": ["lustre_ost"], "file_format": ".h5"},
        "NPZ_64bs": {"scenarios": ["lustre_ost"], "file_format": ".npz"},
        "TFRecord_256kts_64bs": {"scenarios": ["lustre_ost"], "file_format": ".tfrecords"},
        "TFRecord_1mts_64bs": {"scenarios": ["lustre_ost"], "file_format": ".tfrecords"},
    },
}

APPLICATION_POS = 1
DATASET_POS = 1
SCENARIO_POS = 1

APPLICATION = list(CAMPAIGNS)[APPLICATION_POS - 1]
DATASET_GROUP = list(CAMPAIGNS[APPLICATION])[DATASET_POS - 1]
SCENARIO = CAMPAIGNS[APPLICATION][DATASET_GROUP]["scenarios"][SCENARIO_POS - 1]
FILE_FORMAT = CAMPAIGNS[APPLICATION][DATASET_GROUP]["file_format"]

# Module 12-specific derived context. These do not alter campaign selection.
FILESYSTEM = infer_filesystem(SCENARIO, "auto")
STRIPE_COUNT = "auto"
STRIPE_SCOPE = resolve_stripe_scope(
    application=APPLICATION,
    scenario=SCENARIO,
    requested=STRIPE_COUNT,
)
ARTICLE_APPLICATION = article_application_name(APPLICATION)

print("Application         :", APPLICATION)
print("Article application :", ARTICLE_APPLICATION)
print("Dataset group       :", DATASET_GROUP)
print("Scenario            :", SCENARIO)
print("File format         :", FILE_FORMAT)
print("Filesystem          :", FILESYSTEM)
print("Stripe scope        :", "MULTIPLE/AUTO" if STRIPE_SCOPE is None else STRIPE_SCOPE)


Application         : DeepGalaxy
Article application : DeepGalaxy
Dataset group       : DG_bw512_f3c5x64
Scenario            : lustre_1ost
File format         : .hdf5
Filesystem          : lustre
Stripe scope        : 1


In [24]:
CAMPAIGN_ROOT = DATA_ROOT / APPLICATION / DATASET_GROUP / SCENARIO
GOLDEN = Path.cwd() / "Config" / "article01_golden_reference.csv"
print("Campaign root:", CAMPAIGN_ROOT)
print("Golden       :", GOLDEN)


Campaign root: F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\lustre_1ost
Golden       : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Scripts\DeepTuneIO\12_Article_Validation\12_Article_Validation_v1_5_2_1\Config\article01_golden_reference.csv


In [25]:
args=[
    "--campaign-root",str(CAMPAIGN_ROOT),
    "--golden-reference",str(GOLDEN),
    "--article-id","Article_01",
    "--application",APPLICATION,
    "--dataset-group",DATASET_GROUP,
    "--scenario",SCENARIO,
    "--file-format",FILE_FORMAT,
    "--filesystem",FILESYSTEM,
    "--stripe-count",STRIPE_COUNT,
    "--module-root",str(MODULE_ROOT),
]
rc=module12_auto.main(args)
print("Return code:",rc)


DeepTuneIO — Module 12: Automatic Campaign Article Validation
Version          : 1.5.2.1
Article          : Article_01
Application      : DeepGalaxy
Article app      : DeepGalaxy
Dataset group    : DG_bw512_f3c5x64
Scenario         : lustre_1ost
File format      : .hdf5
Filesystem       : lustre
Stripe scope     : 1
Campaign root    : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Data\DeepGalaxy\DG_bw512_f3c5x64\lustre_1ost
Golden reference : F:\NoteBooks\CORA_DR_UAB\Article_01_Parallel_IO_Analysis\Scripts\DeepTuneIO\12_Article_Validation\12_Article_Validation_v1_5_2_1\Config\article01_golden_reference.csv

Resolved directly from Golden Reference:
 figure          article_mode    result_access_mode stripe_counts
     11                shared                shared           [1]
     12 shared_reload_shuffle shared_reload_shuffle           [1]

Resolved structural/visual Article references:
 figure panel                logical_figure  processes data_loading_mode
     10     a 

If any figure is `FAIL`, `INCOMPLETE` or `ERROR`, execute the provenance diagnostic. Do not change tolerances or references to force PASS.